In [1]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

TEMP_DIR = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "climate"
    / "temperature"
)

TMAX_DIR = TEMP_DIR / "Tmax"
TMIN_DIR = TEMP_DIR / "Tmin"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("TMAX_DIR:", TMAX_DIR)
print("TMIN_DIR:", TMIN_DIR)

PROJECT_ROOT: c:\Users\nonoe\OneDrive\Escritorio\Proyectos\AgroCebada-FIRA-2026
TMAX_DIR: c:\Users\nonoe\OneDrive\Escritorio\Proyectos\AgroCebada-FIRA-2026\data\raw\climate\temperature\Tmax
TMIN_DIR: c:\Users\nonoe\OneDrive\Escritorio\Proyectos\AgroCebada-FIRA-2026\data\raw\climate\temperature\Tmin


In [2]:
tmax_files = sorted(TMAX_DIR.rglob("*.tif"))
tmin_files = sorted(TMIN_DIR.rglob("*.tif"))

print("Archivos Tmax:", len(tmax_files))
print("Archivos Tmin:", len(tmin_files))
print("Total:", len(tmax_files) + len(tmin_files))

Archivos Tmax: 48
Archivos Tmin: 48
Total: 96


In [3]:
import rasterio

# Primer raster Tmax
with rasterio.open(tmax_files[0]) as src:
    print("=== Tmax ===")
    print("Archivo:", tmax_files[0].name)
    print("CRS:", src.crs)
    print("Resolución:", src.res)
    print("Dimensiones:", src.width, "x", src.height)
    print("Bandas:", src.count)
    print("NoData:", src.nodata)
    print("Tipo de dato:", src.dtypes[0])
    print("Bounds:", src.bounds)

# Primer raster Tmin
with rasterio.open(tmin_files[0]) as src:
    print("\n=== Tmin ===")
    print("Archivo:", tmin_files[0].name)
    print("CRS:", src.crs)
    print("Resolución:", src.res)
    print("Dimensiones:", src.width, "x", src.height)
    print("Bandas:", src.count)
    print("NoData:", src.nodata)
    print("Tipo de dato:", src.dtypes[0])
    print("Bounds:", src.bounds)

=== Tmax ===
Archivo: Tmax_2022_01.tif
CRS: EPSG:4326
Resolución: (0.05000000074505806, 0.05000000074505806)
Dimensiones: 65 x 72
Bandas: 1
NoData: -9999.0
Tipo de dato: float32
Bounds: BoundingBox(left=-99.94999880716205, bottom=17.799999222159386, right=-96.69999875873327, top=21.399999275803566)

=== Tmin ===
Archivo: Tmin_2022_01.tif
CRS: EPSG:4326
Resolución: (0.05000000074505806, 0.05000000074505806)
Dimensiones: 65 x 72
Bandas: 1
NoData: -9999.0
Tipo de dato: float32
Bounds: BoundingBox(left=-99.94999880716205, bottom=17.799999222159386, right=-96.69999875873327, top=21.399999275803566)


In [4]:
def revisar_consistencia_rasters(files):
    crs_set = set()
    res_set = set()
    dims_set = set()
    nodata_set = set()
    dtype_set = set()
    bounds_set = set()

    for path in files:
        with rasterio.open(path) as src:
            crs_set.add(str(src.crs))
            res_set.add(src.res)
            dims_set.add((src.width, src.height))
            nodata_set.add(src.nodata)
            dtype_set.add(src.dtypes[0])
            bounds_set.add(tuple(src.bounds))

    return {
        "CRS": crs_set,
        "Resoluciones": res_set,
        "Dimensiones": dims_set,
        "NoData": nodata_set,
        "Tipos de dato": dtype_set,
        "Bounds únicos": len(bounds_set),
    }


print("=== CONSISTENCIA TMAX ===")
print(revisar_consistencia_rasters(tmax_files))

print("\n=== CONSISTENCIA TMIN ===")
print(revisar_consistencia_rasters(tmin_files))

=== CONSISTENCIA TMAX ===
{'CRS': {'EPSG:4326'}, 'Resoluciones': {(0.05000000074505806, 0.05000000074505806)}, 'Dimensiones': {(65, 72)}, 'NoData': {-9999.0}, 'Tipos de dato': {'float32'}, 'Bounds únicos': 1}

=== CONSISTENCIA TMIN ===
{'CRS': {'EPSG:4326'}, 'Resoluciones': {(0.05000000074505806, 0.05000000074505806)}, 'Dimensiones': {(65, 72)}, 'NoData': {-9999.0}, 'Tipos de dato': {'float32'}, 'Bounds únicos': 1}


In [5]:
import re

def extraer_periodos(files):
    periodos = []

    for path in files:
        match = re.search(r"_(\d{4})_(\d{2})\.tif$", path.name)

        if match:
            year = int(match.group(1))
            month = int(match.group(2))
            periodos.append((year, month))

    return sorted(periodos)


tmax_periodos = extraer_periodos(tmax_files)
tmin_periodos = extraer_periodos(tmin_files)

print("=== TMAX ===")
print("Primer periodo:", tmax_periodos[0])
print("Último periodo:", tmax_periodos[-1])
print("Periodos únicos:", len(set(tmax_periodos)))
print("Periodos faltantes:", set(tmax_periodos) ^ set(
    [(year, month) for year in range(2022, 2026) for month in range(1, 13)]
))

print("\n=== TMIN ===")
print("Primer periodo:", tmin_periodos[0])
print("Último periodo:", tmin_periodos[-1])
print("Periodos únicos:", len(set(tmin_periodos)))
print("Periodos faltantes:", set(tmin_periodos) ^ set(
    [(year, month) for year in range(2022, 2026) for month in range(1, 13)]
))


=== TMAX ===
Primer periodo: (2022, 1)
Último periodo: (2025, 12)
Periodos únicos: 48
Periodos faltantes: set()

=== TMIN ===
Primer periodo: (2022, 1)
Último periodo: (2025, 12)
Periodos únicos: 48
Periodos faltantes: set()


In [6]:
import numpy as np

for label, path in [
    ("Tmax", tmax_files[0]),
    ("Tmin", tmin_files[0])
]:
    with rasterio.open(path) as src:
        data = src.read(1)
        nodata = src.nodata

        valid = data[data != nodata]

        print(f"\n=== {label} ===")
        print("Archivo:", path.name)
        print("Mínimo válido:", valid.min())
        print("Máximo válido:", valid.max())
        print("Media válida:", valid.mean())
        print("Mediana válida:", np.median(valid))
        print("Píxeles válidos:", len(valid))
        print("Píxeles NoData:", np.sum(data == nodata))


=== Tmax ===
Archivo: Tmax_2022_01.tif
Mínimo válido: 19.663115
Máximo válido: 29.675657
Media válida: 23.698545
Mediana válida: 23.334251
Píxeles válidos: 4462
Píxeles NoData: 218

=== Tmin ===
Archivo: Tmin_2022_01.tif
Mínimo válido: 5.6472836
Máximo válido: 21.161032
Media válida: 11.354049
Mediana válida: 10.972631
Píxeles válidos: 4462
Píxeles NoData: 218


In [7]:
def auditar_valores_temperatura(files):
    resultados = []

    for path in files:
        with rasterio.open(path) as src:
            data = src.read(1)
            nodata = src.nodata

            valid = data[data != nodata]

            resultados.append({
                "archivo": path.name,
                "min": float(valid.min()),
                "max": float(valid.max()),
                "media": float(valid.mean()),
                "mediana": float(np.median(valid)),
                "validos": int(len(valid)),
                "nodata": int(np.sum(data == nodata)),
            })

    return resultados


tmax_auditoria = auditar_valores_temperatura(tmax_files)
tmin_auditoria = auditar_valores_temperatura(tmin_files)

print("Auditoría Tmax:", len(tmax_auditoria), "archivos")
print("Auditoría Tmin:", len(tmin_auditoria), "archivos")

Auditoría Tmax: 48 archivos
Auditoría Tmin: 48 archivos


In [8]:
import pandas as pd

tmax_auditoria_df = pd.DataFrame(tmax_auditoria)
tmin_auditoria_df = pd.DataFrame(tmin_auditoria)

print("=== RANGO GLOBAL TMAX ===")
print("Mínimo:", tmax_auditoria_df["min"].min())
print("Máximo:", tmax_auditoria_df["max"].max())

print("\n=== RANGO GLOBAL TMIN ===")
print("Mínimo:", tmin_auditoria_df["min"].min())
print("Máximo:", tmin_auditoria_df["max"].max())

=== RANGO GLOBAL TMAX ===
Mínimo: 17.05921745300293
Máximo: 41.97311019897461

=== RANGO GLOBAL TMIN ===
Mínimo: 4.773643493652344
Máximo: 33.944026947021484


In [10]:
print("=== TMAX extremos ===")

print(
    tmax_auditoria_df.loc[
        tmax_auditoria_df["min"].idxmin()
    ]
)

print(
    tmax_auditoria_df.loc[
        tmax_auditoria_df["max"].idxmax()
    ]
)

print("\n=== TMIN extremos ===")

print(
    tmin_auditoria_df.loc[
        tmin_auditoria_df["min"].idxmin()
    ]
)

print(
    tmin_auditoria_df.loc[
        tmin_auditoria_df["max"].idxmax()
    ]
)

=== TMAX extremos ===
archivo    Tmax_2022_09.tif
min               17.059217
max                33.08242
media             25.345301
mediana           24.673084
validos                4462
nodata                  218
Name: 8, dtype: object
archivo    Tmax_2024_05.tif
min               28.331556
max                41.97311
media             33.668949
mediana           33.634346
validos                4462
nodata                  218
Name: 28, dtype: object

=== TMIN extremos ===
archivo    Tmin_2023_01.tif
min                4.773643
max               23.991819
media             11.305245
mediana           10.429208
validos                4462
nodata                  218
Name: 12, dtype: object
archivo    Tmin_2023_06.tif
min               10.559864
max               33.944027
media             18.768013
mediana           17.939663
validos                4462
nodata                  218
Name: 17, dtype: object


In [11]:
inconsistencias_temperatura = []

for tmax_path, tmin_path in zip(tmax_files, tmin_files):

    with rasterio.open(tmax_path) as src_max:
        tmax = src_max.read(1)
        nodata_max = src_max.nodata

    with rasterio.open(tmin_path) as src_min:
        tmin = src_min.read(1)
        nodata_min = src_min.nodata

    # Píxeles donde ambos tienen datos válidos
    validos = (
        (tmax != nodata_max) &
        (tmin != nodata_min) &
        np.isfinite(tmax) &
        np.isfinite(tmin)
    )

    # Casos físicamente inconsistentes
    inconsistentes = validos & (tmax < tmin)

    cantidad = int(np.sum(inconsistentes))

    if cantidad > 0:
        inconsistencias_temperatura.append({
            "archivo_tmax": tmax_path.name,
            "archivo_tmin": tmin_path.name,
            "pixeles_inconsistentes": cantidad,
            "diferencia_minima": float(
                np.min(tmax[inconsistentes] - tmin[inconsistentes])
            )
        })

print(
    "Meses con Tmax < Tmin:",
    len(inconsistencias_temperatura)
)

print(
    inconsistencias_temperatura
)

Meses con Tmax < Tmin: 0
[]


In [12]:
import geopandas as gpd
import rasterio
import numpy as np

PARCELS_PATH = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "parcels"
    / "Parcelas_Reto_AGC_CONJUNTO.shp"
)

parcels = gpd.read_file(PARCELS_PATH)

print("CRS parcelas:", parcels.crs)
print("Número de parcelas:", len(parcels))

print("\nCRS Tmax:", rasterio.open(tmax_files[0]).crs)
print("CRS Tmin:", rasterio.open(tmin_files[0]).crs)

CRS parcelas: EPSG:4326
Número de parcelas: 197

CRS Tmax: EPSG:4326
CRS Tmin: EPSG:4326


In [14]:
from rasterio.mask import mask

In [15]:
def revisar_cobertura_raster(parcels, raster_files):
    resultados = []

    for path in raster_files:
        with rasterio.open(path) as src:

            nodata = src.nodata
            data = src.read(1)

            for _, parcela in parcels.iterrows():

                geom = [parcela.geometry]

                # Máscara de los píxeles que intersectan la parcela
                out_image, _ = rasterio.mask.mask(
                    src,
                    geom,
                    crop=True,
                    filled=False
                )

                valores = out_image[0]

                # Valores válidos dentro de la parcela
                validos = valores.compressed()

                resultados.append({
                    "archivo": path.name,
                    "ID_POLIGON": parcela["ID_POLIGON"],
                    "pixeles_validos": len(validos),
                    "tiene_datos": len(validos) > 0
                })

    return pd.DataFrame(resultados)

In [16]:
cobertura_tmax = revisar_cobertura_raster(
    parcels,
    tmax_files
)

cobertura_tmin = revisar_cobertura_raster(
    parcels,
    tmin_files
)

print("Tmax:", cobertura_tmax.shape)
print("Tmin:", cobertura_tmin.shape)

Tmax: (9456, 4)
Tmin: (9456, 4)


In [17]:
print("=== TMAX ===")
print(
    cobertura_tmax.groupby("archivo")["tiene_datos"]
    .sum()
    .describe()
)

print("\nParcelas sin datos Tmax:")
print(
    (~cobertura_tmax["tiene_datos"]).sum()
)

print("\n=== TMIN ===")
print(
    cobertura_tmin.groupby("archivo")["tiene_datos"]
    .sum()
    .describe()
)

print("\nParcelas sin datos Tmin:")
print(
    (~cobertura_tmin["tiene_datos"]).sum()
)

=== TMAX ===
count    48.0
mean      1.0
std       0.0
min       1.0
25%       1.0
50%       1.0
75%       1.0
max       1.0
Name: tiene_datos, dtype: float64

Parcelas sin datos Tmax:
9408

=== TMIN ===
count    48.0
mean      1.0
std       0.0
min       1.0
25%       1.0
50%       1.0
75%       1.0
max       1.0
Name: tiene_datos, dtype: float64

Parcelas sin datos Tmin:
9408


In [18]:
def resumen_cobertura_por_mes(cobertura_df):
    resumen = (
        cobertura_df
        .groupby("archivo")
        .agg(
            parcelas_total=("ID_POLIGON", "nunique"),
            parcelas_con_datos=("tiene_datos", "sum"),
            parcelas_sin_datos=("tiene_datos", lambda x: (~x).sum()),
        )
        .reset_index()
    )

    return resumen


resumen_tmax = resumen_cobertura_por_mes(cobertura_tmax)
resumen_tmin = resumen_cobertura_por_mes(cobertura_tmin)

print("=== COBERTURA TMAX ===")
print(resumen_tmax)

print("\n=== COBERTURA TMIN ===")
print(resumen_tmin)

=== COBERTURA TMAX ===
             archivo  parcelas_total  parcelas_con_datos  parcelas_sin_datos
0   Tmax_2022_01.tif             197                   1                 196
1   Tmax_2022_02.tif             197                   1                 196
2   Tmax_2022_03.tif             197                   1                 196
3   Tmax_2022_04.tif             197                   1                 196
4   Tmax_2022_05.tif             197                   1                 196
5   Tmax_2022_06.tif             197                   1                 196
6   Tmax_2022_07.tif             197                   1                 196
7   Tmax_2022_08.tif             197                   1                 196
8   Tmax_2022_09.tif             197                   1                 196
9   Tmax_2022_10.tif             197                   1                 196
10  Tmax_2022_11.tif             197                   1                 196
11  Tmax_2022_12.tif             197                 

In [19]:
# Elegimos el primer raster Tmax
test_raster = tmax_files[0]

# Elegimos las primeras 5 parcelas
parcelas_test = parcels.iloc[:5]

with rasterio.open(test_raster) as src:

    print("Raster:")
    print("  CRS:", src.crs)
    print("  Bounds:", src.bounds)
    print("  Shape:", src.shape)
    print("  NoData:", src.nodata)

    print("\nParcelas:")

    for _, parcela in parcelas_test.iterrows():

        geom = [parcela.geometry]

        try:
            out_image, out_transform = mask(
                src,
                geom,
                crop=True,
                filled=False
            )

            valores = out_image[0]
            validos = valores.compressed()

            print(
                parcela["ID_POLIGON"],
                "| validos:", len(validos),
                "| shape:", valores.shape,
                "| bounds parcela:", parcela.geometry.bounds
            )

        except Exception as e:

            print(
                parcela["ID_POLIGON"],
                "| ERROR:",
                type(e).__name__,
                "|",
                str(e)
            )

Raster:
  CRS: EPSG:4326
  Bounds: BoundingBox(left=-99.94999880716205, bottom=17.799999222159386, right=-96.69999875873327, top=21.399999275803566)
  Shape: (72, 65)
  NoData: -9999.0

Parcelas:
AGC_196 | validos: 0 | shape: (1, 1) | bounds parcela: (-98.580992, 19.65494, -98.577391, 19.658024)
AGC_044 | validos: 0 | shape: (1, 1) | bounds parcela: (-98.587004, 19.628597, -98.58557, 19.63014)
AGC_020 | validos: 0 | shape: (1, 1) | bounds parcela: (-98.587973, 19.625947, -98.584216, 19.629825)
AGC_174 | validos: 0 | shape: (2, 1) | bounds parcela: (-98.3060387099906, 19.8967442081859, -98.3018908541506, 19.8999999535217)
AGC_083 | validos: 0 | shape: (1, 1) | bounds parcela: (-98.2938319509557, 19.8872213595361, -98.2912989812306, 19.8893702073677)


In [21]:
test_raster = tmax_files[0]

parcela = parcels.loc[
    parcels["ID_POLIGON"] == "AGC_196"
].iloc[0]

with rasterio.open(test_raster) as src:

    out_image, out_transform = mask(
        src,
        [parcela.geometry],
        crop=True,
        filled=False
    )

    valores = out_image[0]

    print("Array:")
    print(valores)

    print("\nMasked:")
    print(np.ma.getmaskarray(valores))

    print("\nData:")
    print(valores.data)

    print("\nShape:", valores.shape)
    print("Número total de celdas:", valores.size)
    print("Número enmascarado:", np.ma.count_masked(valores))
    print("Número válido:", np.ma.count(valores))

Array:
[[--]]

Masked:
[[ True]]

Data:
[[20.394972]]

Shape: (1, 1)
Número total de celdas: 1
Número enmascarado: 1
Número válido: 0


In [22]:
test_raster = tmax_files[0]

parcela = parcels.loc[
    parcels["ID_POLIGON"] == "AGC_196"
].iloc[0]

with rasterio.open(test_raster) as src:

    out_image, out_transform = mask(
        src,
        [parcela.geometry],
        crop=True,
        all_touched=True,
        filled=False
    )

    valores = out_image[0]

    print("Valores all_touched:")
    print(valores)

    print("\nValores válidos:")
    print(valores.compressed())

    print("\nNúmero de píxeles válidos:")
    print(np.ma.count(valores))

Valores all_touched:
[[20.39497184753418]]

Valores válidos:
[20.394972]

Número de píxeles válidos:
1


In [23]:
from shapely.geometry import Point

test_raster = tmax_files[0]

parcela = parcels.loc[
    parcels["ID_POLIGON"] == "AGC_196"
].iloc[0]

centroide = parcela.geometry.centroid

with rasterio.open(test_raster) as src:

    fila, columna = src.index(
        centroide.x,
        centroide.y
    )

    valor_centroid = src.read(1)[fila, columna]

    print("Centroide:", centroide.x, centroide.y)
    print("Fila:", fila)
    print("Columna:", columna)
    print("Tmax centroide:", valor_centroid)

Centroide: -98.57907446800543 19.656256748865168
Fila: 34
Columna: 27
Tmax centroide: 20.394972


In [24]:
import geopandas as gpd
from shapely.geometry import box

test_raster = tmax_files[0]

parcela = parcels.loc[
    parcels["ID_POLIGON"] == "AGC_196"
].iloc[0]

# CRS métrico que ya usamos para la auditoría topográfica/CHIRPS
parcela_metric = (
    gpd.GeoSeries(
        [parcela.geometry],
        crs=parcels.crs
    )
    .to_crs("EPSG:6372")
    .iloc[0]
)

with rasterio.open(test_raster) as src:

    # Centroide del píxel donde cae el centroide de la parcela
    centroide = parcela.geometry.centroid

    fila, columna = src.index(
        centroide.x,
        centroide.y
    )

    # Límites del píxel
    pixel_bounds = src.window_bounds(
        rasterio.windows.Window(
            columna,
            fila,
            1,
            1
        )
    )

    pixel_geom = box(*pixel_bounds)

    # Transformar píxel a CRS métrico
    pixel_metric = (
        gpd.GeoSeries(
            [pixel_geom],
            crs=src.crs
        )
        .to_crs("EPSG:6372")
        .iloc[0]
    )

    # Intersección
    interseccion = parcela_metric.intersection(pixel_metric)

    area_parcela = parcela_metric.area
    area_interseccion = interseccion.area

    peso = area_interseccion / area_parcela

    valor_pixel = src.read(1)[fila, columna]

    temperatura_ponderada = valor_pixel * peso

    print("Tmax del píxel:", valor_pixel)
    print("Área parcela (m²):", area_parcela)
    print("Área intersección (m²):", area_interseccion)
    print("Peso:", peso)
    print("Tmax ponderada:", temperatura_ponderada)

Tmax del píxel: 20.394972
Área parcela (m²): 85130.66878760715
Área intersección (m²): 85130.66878760715
Peso: 1.0
Tmax ponderada: 20.394972


In [25]:
test_raster = tmax_files[0]

parcelas_multipixel = []

with rasterio.open(test_raster) as src:

    for _, parcela in parcels.iterrows():

        geom = [parcela.geometry]

        out_image, _ = mask(
            src,
            geom,
            crop=True,
            all_touched=True,
            filled=False
        )

        n_pixeles = np.ma.count(out_image[0])

        if n_pixeles > 1:
            parcelas_multipixel.append({
                "ID_POLIGON": parcela["ID_POLIGON"],
                "n_pixeles": int(n_pixeles),
                "area_ha": parcela["área_ha"]
            })

multipixel_df = pd.DataFrame(parcelas_multipixel)

print("Parcelas que tocan >1 píxel:", len(multipixel_df))
print()
print(multipixel_df.sort_values("n_pixeles", ascending=False).head(10))

Parcelas que tocan >1 píxel: 31

   ID_POLIGON  n_pixeles  area_ha
5     AGC_180          3    10.90
25    AGC_123          3     8.01
1     AGC_084          2    12.42
2     AGC_051          2    11.12
3     AGC_059          2    19.53
4     AGC_082          2     2.77
0     AGC_174          2     6.89
7     AGC_041          2    25.13
8     AGC_042          2     3.81
9     AGC_194          2     4.12


In [26]:
from rasterio.mask import mask
from rasterio.transform import rowcol
from shapely.geometry import box

test_raster = tmax_files[0]

parcela = parcels.loc[
    parcels["ID_POLIGON"] == "AGC_180"
].iloc[0]

# ---------------------------------------------------------
# 1. CENTROIDE
# ---------------------------------------------------------

centroide = parcela.geometry.centroid

with rasterio.open(test_raster) as src:

    fila, columna = src.index(
        centroide.x,
        centroide.y
    )

    valor_centroid = float(
        src.read(1)[fila, columna]
    )

# ---------------------------------------------------------
# 2. ALL TOUCHED
# ---------------------------------------------------------

with rasterio.open(test_raster) as src:

    out_image, _ = mask(
        src,
        [parcela.geometry],
        crop=True,
        all_touched=True,
        filled=False
    )

    valores_all = out_image[0].compressed()

valor_all_touched = float(np.mean(valores_all))

# ---------------------------------------------------------
# 3. ÁREA PONDERADA
# ---------------------------------------------------------

parcela_metric = (
    gpd.GeoSeries(
        [parcela.geometry],
        crs=parcels.crs
    )
    .to_crs("EPSG:6372")
    .iloc[0]
)

with rasterio.open(test_raster) as src:

    # Transformamos la parcela a coordenadas del raster
    bounds = parcela.geometry.bounds

    # Determinamos ventana de píxeles candidatos
    row_min, col_min = src.index(bounds[0], bounds[3])
    row_max, col_max = src.index(bounds[2], bounds[1])

    # Aseguramos orden correcto
    row_start = min(row_min, row_max)
    row_end = max(row_min, row_max)

    col_start = min(col_min, col_max)
    col_end = max(col_min, col_max)

    resultados = []

    for row in range(row_start, row_end + 1):
        for col in range(col_start, col_end + 1):

            valor = src.read(
                1,
                window=rasterio.windows.Window(
                    col,
                    row,
                    1,
                    1
                )
            )[0, 0]

            if value := np.isfinite(valor):
                pass

            if valor == src.nodata:
                continue

            pixel_bounds = src.window_bounds(
                rasterio.windows.Window(
                    col,
                    row,
                    1,
                    1
                )
            )

            pixel_geom = box(*pixel_bounds)

            pixel_metric = (
                gpd.GeoSeries(
                    [pixel_geom],
                    crs=src.crs
                )
                .to_crs("EPSG:6372")
                .iloc[0]
            )

            interseccion = parcela_metric.intersection(
                pixel_metric
            )

            area_interseccion = interseccion.area

            if area_interseccion > 0:

                resultados.append({
                    "row": row,
                    "col": col,
                    "temperatura": float(valor),
                    "area_interseccion_m2": area_interseccion
                })

df_pixeles = pd.DataFrame(resultados)

df_pixeles["peso"] = (
    df_pixeles["area_interseccion_m2"]
    / df_pixeles["area_interseccion_m2"].sum()
)

valor_area_weighted = float(
    np.sum(
        df_pixeles["temperatura"]
        * df_pixeles["peso"]
    )
)

# ---------------------------------------------------------
# RESULTADOS
# ---------------------------------------------------------

print("PARCELA:", parcela["ID_POLIGON"])
print("ÁREA:", parcela["área_ha"], "ha")

print("\n=== COMPARACIÓN ===")
print("Centroide:       ", valor_centroid)
print("All touched:     ", valor_all_touched)
print("Área ponderada:  ", valor_area_weighted)

print("\n=== PÍXELES ===")
print(df_pixeles)

print("\nSuma pesos:", df_pixeles["peso"].sum())

PARCELA: AGC_180
ÁREA: 10.9 ha

=== COMPARACIÓN ===
Centroide:        20.0863094329834
All touched:      20.025739669799805
Área ponderada:   20.04776111201801

=== PÍXELES ===
   row  col  temperatura  area_interseccion_m2      peso
0   32   31    20.044743          25810.589782  0.238306
1   33   31    20.086309          60361.255555  0.557308
2   33   32    19.946169          22136.687602  0.204385

Suma pesos: 1.0


In [27]:
from shapely.geometry import box
from shapely.strtree import STRtree
import geopandas as gpd
import rasterio
import numpy as np
import pandas as pd


def construir_pixeles_raster(path_raster, crs_metric="EPSG:6372"):
    """
    Construye las geometrías de todos los píxeles del raster
    y un índice espacial STRtree.

    Las geometrías se transforman a un CRS métrico
    para poder calcular áreas de intersección.
    """

    with rasterio.open(path_raster) as src:

        width = src.width
        height = src.height
        transform = src.transform
        crs_raster = src.crs

        geometries = []
        filas = []
        columnas = []

        for row in range(height):
            for col in range(width):

                bounds = rasterio.windows.bounds(
                    rasterio.windows.Window(
                        col,
                        row,
                        1,
                        1
                    ),
                    transform
                )

                geom = box(*bounds)

                geometries.append(geom)
                filas.append(row)
                columnas.append(col)

    # Convertir geometrías a GeoSeries
    pixels_gdf = gpd.GeoDataFrame(
        {
            "row": filas,
            "col": columnas,
        },
        geometry=geometries,
        crs=crs_raster
    )

    pixels_metric = pixels_gdf.to_crs(crs_metric)

    geometries_metric = list(pixels_metric.geometry)

    tree = STRtree(geometries_metric)

    return pixels_metric, geometries_metric, tree

In [28]:
TMAX_TEST = tmax_files[0]

pixels_gdf, pixel_geometries, pixel_tree = (
    construir_pixeles_raster(TMAX_TEST)
)

print("Número de píxeles:", len(pixels_gdf))
print("CRS raster:", pixels_gdf.crs)
print("Geometrías:", len(pixel_geometries))
print("Árbol espacial creado:", pixel_tree is not None)

Número de píxeles: 4680
CRS raster: EPSG:6372
Geometrías: 4680
Árbol espacial creado: True


In [29]:
def extraer_temperatura_comparacion(
    raster_path,
    parcels,
    pixels_gdf,
    pixel_geometries,
    pixel_tree,
    metric_crs="EPSG:6372"
):
    """
    Extrae temperatura parcelaria mediante:

    1. Centroide
    2. All touched
    3. Área ponderada

    Devuelve una fila por parcela.
    """

    with rasterio.open(raster_path) as src:

        data = src.read(1)
        nodata = src.nodata

        # -------------------------------------------------
        # Centroide
        # -------------------------------------------------

        centroid_values = []

        # -------------------------------------------------
        # Transformar parcelas a CRS métrico
        # -------------------------------------------------

        parcels_metric = parcels.to_crs(metric_crs)

        # -------------------------------------------------
        # Resultados
        # -------------------------------------------------

        resultados = []

        for idx, parcela in parcels.iterrows():

            geom_original = parcela.geometry
            geom_metric = parcels_metric.loc[idx].geometry

            # =============================================
            # 1. CENTROIDE
            # =============================================

            centroid = geom_original.centroid

            row, col = src.index(
                centroid.x,
                centroid.y
            )

            valor_centroid = data[row, col]

            if nodata is not None and valor_centroid == nodata:
                valor_centroid = np.nan

            # =============================================
            # 2. PIXELES INTERSECTADOS
            # =============================================

            candidatos = pixel_tree.query(
                geom_metric
            )

            temperaturas = []
            areas = []
            n_pixeles = 0

            for candidate_idx in candidatos:

                pixel_geom = pixel_geometries[candidate_idx]

                inter = geom_metric.intersection(
                    pixel_geom
                )

                if inter.is_empty:
                    continue

                area_inter = inter.area

                if area_inter <= 0:
                    continue

                row_pixel = pixels_gdf.iloc[candidate_idx]["row"]
                col_pixel = pixels_gdf.iloc[candidate_idx]["col"]

                valor = data[
                    int(row_pixel),
                    int(col_pixel)
                ]

                if nodata is not None and valor == nodata:
                    continue

                if not np.isfinite(valor):
                    continue

                temperaturas.append(float(valor))
                areas.append(float(area_inter))
                n_pixeles += 1

            # =============================================
            # ALL TOUCHED
            # =============================================

            if len(temperaturas) > 0:
                valor_all_touched = float(
                    np.mean(temperaturas)
                )
            else:
                valor_all_touched = np.nan

            # =============================================
            # ÁREA PONDERADA
            # =============================================

            if len(temperaturas) > 0:

                areas = np.array(areas)

                pesos = areas / areas.sum()

                valor_area_weighted = float(
                    np.sum(
                        np.array(temperaturas) * pesos
                    )
                )

                suma_pesos = float(pesos.sum())

            else:

                valor_area_weighted = np.nan
                suma_pesos = np.nan

            resultados.append({
                "ID_POLIGON": parcela["ID_POLIGON"],
                "centroid": valor_centroid,
                "all_touched": valor_all_touched,
                "area_weighted": valor_area_weighted,
                "n_pixeles": n_pixeles,
                "suma_pesos": suma_pesos,
            })

    return pd.DataFrame(resultados)

In [30]:
tmax_2022_01 = extraer_temperatura_comparacion(
    tmax_files[0],
    parcels,
    pixels_gdf,
    pixel_geometries,
    pixel_tree
)

print(tmax_2022_01.shape)

print(
    tmax_2022_01.head()
)

(197, 6)
  ID_POLIGON   centroid  all_touched  area_weighted  n_pixeles  suma_pesos
0    AGC_196  20.394972    20.394972      20.394972          1         1.0
1    AGC_044  20.436686    20.436686      20.436686          1         1.0
2    AGC_020  20.436686    20.436686      20.436686          1         1.0
3    AGC_174  20.226089    20.226089      20.226089          1         1.0
4    AGC_083  20.155764    20.155764      20.155764          1         1.0


In [31]:
print("Filas:", len(tmax_2022_01))
print(
    "IDs únicos:",
    tmax_2022_01["ID_POLIGON"].nunique()
)

print(
    "\nNulos:"
)

print(
    tmax_2022_01[
        [
            "centroid",
            "all_touched",
            "area_weighted"
        ]
    ].isna().sum()
)

print(
    "\nError máximo en suma de pesos:"
)

print(
    np.max(
        np.abs(
            tmax_2022_01["suma_pesos"] - 1
        )
    )
)

Filas: 197
IDs únicos: 197

Nulos:
centroid         0
all_touched      0
area_weighted    0
dtype: int64

Error máximo en suma de pesos:
1.1102230246251565e-16


In [32]:
tmax_comparacion = []

for path in tmax_files:

    print("Procesando:", path.name)

    resultado = extraer_temperatura_comparacion(
        path,
        parcels,
        pixels_gdf,
        pixel_geometries,
        pixel_tree
    )

    # Extraer año y mes del nombre
    partes = path.stem.split("_")

    year = int(partes[1])
    month = int(partes[2])

    resultado["year"] = year
    resultado["month"] = month

    tmax_comparacion.append(resultado)

tmax_comparacion_df = pd.concat(
    tmax_comparacion,
    ignore_index=True
)

print("\nShape final:", tmax_comparacion_df.shape)

Procesando: Tmax_2022_01.tif
Procesando: Tmax_2022_02.tif
Procesando: Tmax_2022_03.tif
Procesando: Tmax_2022_04.tif
Procesando: Tmax_2022_05.tif
Procesando: Tmax_2022_06.tif
Procesando: Tmax_2022_07.tif
Procesando: Tmax_2022_08.tif
Procesando: Tmax_2022_09.tif
Procesando: Tmax_2022_10.tif
Procesando: Tmax_2022_11.tif
Procesando: Tmax_2022_12.tif
Procesando: Tmax_2023_01.tif
Procesando: Tmax_2023_02.tif
Procesando: Tmax_2023_03.tif
Procesando: Tmax_2023_04.tif
Procesando: Tmax_2023_05.tif
Procesando: Tmax_2023_06.tif
Procesando: Tmax_2023_07.tif
Procesando: Tmax_2023_08.tif
Procesando: Tmax_2023_09.tif
Procesando: Tmax_2023_10.tif
Procesando: Tmax_2023_11.tif
Procesando: Tmax_2023_12.tif
Procesando: Tmax_2024_01.tif
Procesando: Tmax_2024_02.tif
Procesando: Tmax_2024_03.tif
Procesando: Tmax_2024_04.tif
Procesando: Tmax_2024_05.tif
Procesando: Tmax_2024_06.tif
Procesando: Tmax_2024_07.tif
Procesando: Tmax_2024_08.tif
Procesando: Tmax_2024_09.tif
Procesando: Tmax_2024_10.tif
Procesando: Tm

In [33]:
print("=== AUDITORÍA TMAX ===")

print(
    "Filas:",
    len(tmax_comparacion_df)
)

print(
    "Parcelas:",
    tmax_comparacion_df["ID_POLIGON"].nunique()
)

print(
    "Periodos:",
    tmax_comparacion_df[
        ["year", "month"]
    ].drop_duplicates().shape[0]
)

print(
    "Duplicados parcela-mes:",
    tmax_comparacion_df.duplicated(
        ["ID_POLIGON", "year", "month"]
    ).sum()
)

print("\nNulos:")
print(
    tmax_comparacion_df[
        [
            "centroid",
            "all_touched",
            "area_weighted"
        ]
    ].isna().sum()
)

print("\nSuma de pesos:")
print(
    "Máximo error:",
    np.max(
        np.abs(
            tmax_comparacion_df["suma_pesos"] - 1
        )
    )
)

print("\nDistribución de píxeles:")
print(
    tmax_comparacion_df["n_pixeles"]
    .value_counts()
    .sort_index()
)

=== AUDITORÍA TMAX ===
Filas: 9456
Parcelas: 197
Periodos: 48
Duplicados parcela-mes: 0

Nulos:
centroid         0
all_touched      0
area_weighted    0
dtype: int64

Suma de pesos:
Máximo error: 1.1102230246251565e-16

Distribución de píxeles:
n_pixeles
1    8016
2    1344
3      96
Name: count, dtype: int64


In [34]:
comparacion_tmax = tmax_comparacion_df.copy()

comparacion_tmax["diff_centroid_area"] = (
    comparacion_tmax["centroid"]
    - comparacion_tmax["area_weighted"]
).abs()

comparacion_tmax["diff_all_area"] = (
    comparacion_tmax["all_touched"]
    - comparacion_tmax["area_weighted"]
).abs()

comparacion_tmax["diff_centroid_all"] = (
    comparacion_tmax["centroid"]
    - comparacion_tmax["all_touched"]
).abs()


print("=== CENTROIDE vs ÁREA PONDERADA ===")
print(
    comparacion_tmax["diff_centroid_area"]
    .describe(
        percentiles=[0.5, 0.75, 0.90, 0.95, 0.99]
    )
)

print("\n=== ALL TOUCHED vs ÁREA PONDERADA ===")
print(
    comparacion_tmax["diff_all_area"]
    .describe(
        percentiles=[0.5, 0.75, 0.90, 0.95, 0.99]
    )
)

print("\n=== CENTROIDE vs ALL TOUCHED ===")
print(
    comparacion_tmax["diff_centroid_all"]
    .describe(
        percentiles=[0.5, 0.75, 0.90, 0.95, 0.99]
    )
)

=== CENTROIDE vs ÁREA PONDERADA ===
count    9456.000000
mean        0.004672
std         0.018663
min         0.000000
50%         0.000000
75%         0.000000
90%         0.007350
95%         0.031673
99%         0.110269
max         0.196105
Name: diff_centroid_area, dtype: float64

=== ALL TOUCHED vs ÁREA PONDERADA ===
count    9456.000000
mean        0.005762
std         0.020341
min         0.000000
50%         0.000000
75%         0.000000
90%         0.015823
95%         0.042373
99%         0.110540
max         0.220203
Name: diff_all_area, dtype: float64

=== CENTROIDE vs ALL TOUCHED ===
count    9456.000000
mean        0.010434
std         0.034651
min         0.000000
50%         0.000000
75%         0.000000
90%         0.032154
95%         0.069530
99%         0.192559
max         0.271256
Name: diff_centroid_all, dtype: float64


In [35]:
print("=== OBSERVACIONES CON DIFERENCIA ===")

print(
    "Centroide vs área > 0.01 °C:",
    (
        comparacion_tmax["diff_centroid_area"] > 0.01
    ).sum()
)

print(
    "Centroide vs área > 0.1 °C:",
    (
        comparacion_tmax["diff_centroid_area"] > 0.1
    ).sum()
)

print(
    "Centroide vs área > 0.5 °C:",
    (
        comparacion_tmax["diff_centroid_area"] > 0.5
    ).sum()
)

print(
    "Centroide vs área > 1.0 °C:",
    (
        comparacion_tmax["diff_centroid_area"] > 1.0
    ).sum()
)

=== OBSERVACIONES CON DIFERENCIA ===
Centroide vs área > 0.01 °C: 881
Centroide vs área > 0.1 °C: 114
Centroide vs área > 0.5 °C: 0
Centroide vs área > 1.0 °C: 0


In [36]:
print(
    comparacion_tmax
    .groupby("n_pixeles")[
        [
            "diff_centroid_area",
            "diff_all_area",
            "diff_centroid_all"
        ]
    ]
    .agg(
        ["mean", "median", "max"]
    )
)

          diff_centroid_area                     diff_all_area            \
                        mean    median       max          mean    median   
n_pixeles                                                                  
1                   0.000000  0.000000  0.000000      0.000000  0.000000   
2                   0.029525  0.014902  0.196105      0.036776  0.025254   
3                   0.046865  0.046267  0.110279      0.052698  0.043763   

                    diff_centroid_all                      
                max              mean    median       max  
n_pixeles                                                  
1          0.000000          0.000000  0.000000  0.000000  
2          0.220203          0.066302  0.045273  0.271256  
3          0.138742          0.099543  0.098321  0.213765  


In [37]:
top_tmax = (
    comparacion_tmax
    .sort_values(
        "diff_centroid_area",
        ascending=False
    )
    .head(10)
)

print(
    top_tmax[
        [
            "ID_POLIGON",
            "year",
            "month",
            "centroid",
            "all_touched",
            "area_weighted",
            "n_pixeles",
            "diff_centroid_area"
        ]
    ].to_string(index=False)
)

ID_POLIGON  year  month  centroid  all_touched  area_weighted  n_pixeles  diff_centroid_area
   AGC_059  2025      8 23.103359    23.374616      23.299464          2            0.196105
   AGC_059  2022      8 22.890114    23.147003      23.075832          2            0.185718
   AGC_026  2023      7 23.660723    23.881863      23.846397          2            0.185674
   AGC_059  2025      9 22.075449    22.332025      22.260940          2            0.185491
   AGC_059  2025      7 22.453531    22.709744      22.638760          2            0.185229
   AGC_026  2023     11 21.879307    22.097376      22.062402          2            0.183096
   AGC_059  2023      8 23.287687    23.540696      23.470600          2            0.182913
   AGC_059  2024      7 23.147135    23.397743      23.328312          2            0.181177
   AGC_059  2024      8 23.304289    23.554376      23.485089          2            0.180800
   AGC_059  2024      9 22.202667    22.451429      22.382510         

In [38]:
resumen_mensual_tmax = (
    comparacion_tmax
    .groupby("month")
    .agg(
        media_diff_centroid_area=(
            "diff_centroid_area",
            "mean"
        ),
        mediana_diff_centroid_area=(
            "diff_centroid_area",
            "median"
        ),
        p95_diff_centroid_area=(
            "diff_centroid_area",
            lambda x: x.quantile(0.95)
        ),
        max_diff_centroid_area=(
            "diff_centroid_area",
            "max"
        )
    )
    .reset_index()
)

print(resumen_mensual_tmax.to_string(index=False))

 month  media_diff_centroid_area  mediana_diff_centroid_area  p95_diff_centroid_area  max_diff_centroid_area
     1                  0.002748                         0.0                0.020650                0.095793
     2                  0.002123                         0.0                0.015462                0.087010
     3                  0.002010                         0.0                0.015374                0.074483
     4                  0.002265                         0.0                0.017591                0.079546
     5                  0.003909                         0.0                0.031934                0.136274
     6                  0.005689                         0.0                0.043405                0.167749
     7                  0.007241                         0.0                0.054011                0.185674
     8                  0.007415                         0.0                0.058824                0.196105
     9             

In [39]:
resumen_anual_tmax = (
    comparacion_tmax
    .groupby("year")
    .agg(
        media_diff_centroid_area=(
            "diff_centroid_area",
            "mean"
        ),
        mediana_diff_centroid_area=(
            "diff_centroid_area",
            "median"
        ),
        p95_diff_centroid_area=(
            "diff_centroid_area",
            lambda x: x.quantile(0.95)
        ),
        max_diff_centroid_area=(
            "diff_centroid_area",
            "max"
        )
    )
    .reset_index()
)

print(resumen_anual_tmax.to_string(index=False))

 year  media_diff_centroid_area  mediana_diff_centroid_area  p95_diff_centroid_area  max_diff_centroid_area
 2022                  0.004617                         0.0                0.031828                0.185718
 2023                  0.004854                         0.0                0.033667                0.185674
 2024                  0.004613                         0.0                0.031530                0.181177
 2025                  0.004606                         0.0                0.028494                0.196105


In [40]:
tmin_comparacion = []

for path in tmin_files:

    print("Procesando:", path.name)

    resultado = extraer_temperatura_comparacion(
        path,
        parcels,
        pixels_gdf,
        pixel_geometries,
        pixel_tree
    )

    partes = path.stem.split("_")

    year = int(partes[1])
    month = int(partes[2])

    resultado["year"] = year
    resultado["month"] = month

    tmin_comparacion.append(resultado)

tmin_comparacion_df = pd.concat(
    tmin_comparacion,
    ignore_index=True
)

print("\nShape final:", tmin_comparacion_df.shape)

Procesando: Tmin_2022_01.tif
Procesando: Tmin_2022_02.tif
Procesando: Tmin_2022_03.tif
Procesando: Tmin_2022_04.tif
Procesando: Tmin_2022_05.tif
Procesando: Tmin_2022_06.tif
Procesando: Tmin_2022_07.tif
Procesando: Tmin_2022_08.tif
Procesando: Tmin_2022_09.tif
Procesando: Tmin_2022_10.tif
Procesando: Tmin_2022_11.tif
Procesando: Tmin_2022_12.tif
Procesando: Tmin_2023_01.tif
Procesando: Tmin_2023_02.tif
Procesando: Tmin_2023_03.tif
Procesando: Tmin_2023_04.tif
Procesando: Tmin_2023_05.tif
Procesando: Tmin_2023_06.tif
Procesando: Tmin_2023_07.tif
Procesando: Tmin_2023_08.tif
Procesando: Tmin_2023_09.tif
Procesando: Tmin_2023_10.tif
Procesando: Tmin_2023_11.tif
Procesando: Tmin_2023_12.tif
Procesando: Tmin_2024_01.tif
Procesando: Tmin_2024_02.tif
Procesando: Tmin_2024_03.tif
Procesando: Tmin_2024_04.tif
Procesando: Tmin_2024_05.tif
Procesando: Tmin_2024_06.tif
Procesando: Tmin_2024_07.tif
Procesando: Tmin_2024_08.tif
Procesando: Tmin_2024_09.tif
Procesando: Tmin_2024_10.tif
Procesando: Tm

In [41]:
print("=== AUDITORÍA TMIN ===")

print("Filas:", len(tmin_comparacion_df))
print(
    "Parcelas:",
    tmin_comparacion_df["ID_POLIGON"].nunique()
)

print(
    "Periodos:",
    tmin_comparacion_df[
        ["year", "month"]
    ].drop_duplicates().shape[0]
)

print(
    "Duplicados parcela-mes:",
    tmin_comparacion_df.duplicated(
        ["ID_POLIGON", "year", "month"]
    ).sum()
)

print("\nNulos:")
print(
    tmin_comparacion_df[
        [
            "centroid",
            "all_touched",
            "area_weighted"
        ]
    ].isna().sum()
)

print("\nMáximo error suma pesos:")

print(
    np.max(
        np.abs(
            tmin_comparacion_df["suma_pesos"] - 1
        )
    )
)

print("\nDistribución de píxeles:")

print(
    tmin_comparacion_df["n_pixeles"]
    .value_counts()
    .sort_index()
)

=== AUDITORÍA TMIN ===
Filas: 9456
Parcelas: 197
Periodos: 48
Duplicados parcela-mes: 0

Nulos:
centroid         0
all_touched      0
area_weighted    0
dtype: int64

Máximo error suma pesos:
1.1102230246251565e-16

Distribución de píxeles:
n_pixeles
1    8016
2    1344
3      96
Name: count, dtype: int64


In [42]:
print("=== AUDITORÍA DE VALORES TMIN ===")

print("\nMínimo global:")
print(tmin_comparacion_df["area_weighted"].min())

print("\nMáximo global:")
print(tmin_comparacion_df["area_weighted"].max())

print("\nMedia:")
print(tmin_comparacion_df["area_weighted"].mean())

print("\nMediana:")
print(tmin_comparacion_df["area_weighted"].median())

print("\nPercentiles:")
print(
    tmin_comparacion_df["area_weighted"].quantile(
        [0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99]
    )
)

print("\nValores negativos:")
print((tmin_comparacion_df["area_weighted"] < 0).sum())

print("\nValores exactamente cero:")
print((tmin_comparacion_df["area_weighted"] == 0).sum())

=== AUDITORÍA DE VALORES TMIN ===

Mínimo global:
5.415943145751953

Máximo global:
16.25983428955078

Media:
11.231370365312195

Mediana:
11.551014512165452

Percentiles:
0.01     5.954758
0.05     7.025712
0.25     9.089084
0.50    11.551015
0.75    13.344477
0.95    14.939937
0.99    15.867823
Name: area_weighted, dtype: float64

Valores negativos:
0

Valores exactamente cero:
0


In [43]:
tmin_comparacion_df["diff_centroid_area"] = (
    tmin_comparacion_df["centroid"]
    - tmin_comparacion_df["area_weighted"]
).abs()

tmin_comparacion_df["diff_all_area"] = (
    tmin_comparacion_df["all_touched"]
    - tmin_comparacion_df["area_weighted"]
).abs()

tmin_comparacion_df["diff_centroid_all"] = (
    tmin_comparacion_df["centroid"]
    - tmin_comparacion_df["all_touched"]
).abs()

print("=== DIFERENCIAS DE MÉTODOS — TMIN ===")

print("\nCentroid vs Area Weighted:")
print(
    tmin_comparacion_df["diff_centroid_area"].describe(
        percentiles=[0.50, 0.75, 0.90, 0.95, 0.99]
    )
)

print("\nAll Touched vs Area Weighted:")
print(
    tmin_comparacion_df["diff_all_area"].describe(
        percentiles=[0.50, 0.75, 0.90, 0.95, 0.99]
    )
)

print("\nCentroid vs All Touched:")
print(
    tmin_comparacion_df["diff_centroid_all"].describe(
        percentiles=[0.50, 0.75, 0.90, 0.95, 0.99]
    )
)

=== DIFERENCIAS DE MÉTODOS — TMIN ===

Centroid vs Area Weighted:
count    9456.000000
mean        0.007514
std         0.026951
min         0.000000
50%         0.000000
75%         0.000000
90%         0.018508
95%         0.056189
99%         0.145883
max         0.267989
Name: diff_centroid_area, dtype: float64

All Touched vs Area Weighted:
count    9456.000000
mean        0.010741
std         0.034133
min         0.000000
50%         0.000000
75%         0.000000
90%         0.035564
95%         0.082823
99%         0.178577
max         0.286772
Name: diff_all_area, dtype: float64

Centroid vs All Touched:
count    9456.000000
mean        0.018254
std         0.053823
min         0.000000
50%         0.000000
75%         0.000000
90%         0.073679
95%         0.136271
99%         0.279617
max         0.370688
Name: diff_centroid_all, dtype: float64


In [44]:
temp_validacion = tmax_comparacion_df[
    ["ID_POLIGON", "year", "month", "area_weighted"]
].merge(
    tmin_comparacion_df[
        ["ID_POLIGON", "year", "month", "area_weighted"]
    ],
    on=["ID_POLIGON", "year", "month"],
    suffixes=("_tmax", "_tmin")
)

print("=== VALIDACIÓN TMAX vs TMIN ===")

print("Filas:", len(temp_validacion))
print("Parcelas:", temp_validacion["ID_POLIGON"].nunique())
print(
    "Periodos:",
    temp_validacion[["year", "month"]].drop_duplicates().shape[0]
)

print("\nCasos Tmax < Tmin:")
print(
    (temp_validacion["area_weighted_tmax"] <
     temp_validacion["area_weighted_tmin"]).sum()
)

print("\nDiferencia Tmax - Tmin:")
print(
    (temp_validacion["area_weighted_tmax"] -
     temp_validacion["area_weighted_tmin"]).describe(
        percentiles=[0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99]
    )
)

=== VALIDACIÓN TMAX vs TMIN ===
Filas: 9456
Parcelas: 197
Periodos: 48

Casos Tmax < Tmin:
0

Diferencia Tmax - Tmin:
count    9456.000000
mean       12.259590
std         2.354039
min         7.452831
1%          7.616676
5%          8.407905
25%        10.178390
50%        12.421607
75%        14.139731
95%        15.826719
99%        17.391541
max        17.810070
dtype: float64


In [45]:
temp_climatologia = (
    temp_validacion
    .groupby("month")
    .agg(
        Tmin_mean=("area_weighted_tmin", "mean"),
        Tmax_mean=("area_weighted_tmax", "mean")
    )
    .reset_index()
)

temp_climatologia["DTR_mean"] = (
    temp_climatologia["Tmax_mean"]
    - temp_climatologia["Tmin_mean"]
)

print(temp_climatologia.to_string(index=False))

 month  Tmin_mean  Tmax_mean  DTR_mean
     1   7.574839  21.271557 13.696718
     2   8.423928  23.128174 14.704246
     3  10.254894  25.613628 15.358734
     4  12.120675  26.114051 13.993376
     5  13.471866  26.726181 13.254315
     6  14.012518  24.663663 10.651145
     7  13.686381  23.532225  9.845844
     8  13.562791  23.632580 10.069790
     9  13.127890  22.671236  9.543346
    10  11.400211  21.896539 10.496327
    11   9.204467  22.486635 13.282169
    12   7.935985  20.155057 12.219072


In [49]:
# ==========================================
# VERIFICACIÓN FINAL DE ARCHIVOS DE TEMPERATURA
# ==========================================

tmin_check = pd.read_csv(tmin_path)
tmax_check = pd.read_csv(tmax_path)

print("=== TMIN ===")
print("Shape:", tmin_check.shape)
print("Parcelas:", tmin_check["ID_POLIGON"].nunique())
print(
    "Periodos:",
    tmin_check[["year", "month"]].drop_duplicates().shape[0]
)
print(
    "Duplicados parcela-mes:",
    tmin_check.duplicated(
        ["ID_POLIGON", "year", "month"]
    ).sum()
)
print("Nulos:")
print(tmin_check.isna().sum())

print("\n=== TMAX ===")
print("Shape:", tmax_check.shape)
print("Parcelas:", tmax_check["ID_POLIGON"].nunique())
print(
    "Periodos:",
    tmax_check[["year", "month"]].drop_duplicates().shape[0]
)
print(
    "Duplicados parcela-mes:",
    tmax_check.duplicated(
        ["ID_POLIGON", "year", "month"]
    ).sum()
)
print("Nulos:")
print(tmax_check.isna().sum())

=== TMIN ===
Shape: (9456, 6)
Parcelas: 197
Periodos: 48
Duplicados parcela-mes: 0
Nulos:
ID_POLIGON    0
year          0
month         0
TMIN_C        0
n_pixeles     0
suma_pesos    0
dtype: int64

=== TMAX ===
Shape: (9456, 6)
Parcelas: 197
Periodos: 48
Duplicados parcela-mes: 0
Nulos:
ID_POLIGON    0
year          0
month         0
TMAX_C        0
n_pixeles     0
suma_pesos    0
dtype: int64
